<a href="https://colab.research.google.com/github/nazaninbondarian/MachineLearning2024/blob/main/Third_Steps/Exponential1/Part1/propofol_td3_induction_exp_reference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Propofol induction with TD3 -- BIS tracks an exponential reference curve (phase 1 only).

Based on Tu et al. 2025 (PK/PD env, TD3) and Jarrett et al. 2022 (exponential reference function RFe).

* Only the **induction phase** is simulated. Each episode ends when the reference reaches the target (t_eps, 4-10 min = 48-120 steps of 5 s).
* Reference: `RFe(t) = (B0 - Bf) * exp(-lambda * t) + Bf`, `Bf` = BIS target, `B0` = 100,
  `lambda = (ln(B0 - Bf) - ln(eps)) / t_eps`, `eps = 1`.
* Reward = BIS tracking (dead-band +-2 around the reference, BIS *below* the reference penalized x2)
  + RR / MAP / HR safety thresholds + infusion-change penalty (free change up to 2 mg/min).
* State: 15-D (12 old inputs + `BIS_ref(t)`, `BIS_ref(t+1)`, `t / t_eps`). Old checkpoints are not compatible.
* Evaluation: tracking error to the reference, RMSE, deepest undershoot, error at the end of induction.

Edit the CONFIG cell (section 7), then run all cells in Colab (token from Colab Secret `GITHUB_TOKEN`).

In [ ]:
from __future__ import annotations

import os

# Must be set before any CUDA context is created (reproducible cuBLAS results).
os.environ.setdefault("PYTHONHASHSEED", "0")

import copy
import getpass
import random
import shutil
import subprocess
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import torch

# FAST TRAINING MODE: prioritize speed over bit-exact CUDA determinism
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True
torch.use_deterministic_algorithms(False)
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
# ============================================================
# Device
# ============================================================
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")

In [ ]:
# =========================================================================== #
# 1) PK/PD environment  (induction only: BIS tracks an exponential reference)
# =========================================================================== #

# Pharmacodynamic constants (paper, Sec. 2.1.2)
BIS0, RR0, MAP0 = 100.0, 20.0, 90.0
EC50_BIS, EC50_RR, EC50_MAP = 4.47, 3.99, 2.2
GAMMA_BIS, GAMMA_RR, GAMMA_MAP = -1.43, -1.43, 1.9

# E_max for MAP is not given in the paper. 1.0 (MAP can drop to 0) is
# physiologically impossible and makes most randomized BIS/MAP target
# combinations unsatisfiable at the same time, so a reduced value is used.
E_MAX_MAP = 0.4

SV = 80.4            # stroke volume (ml)
TPR = 0.018          # total peripheral resistance (mmHg * ml^-1 * min)

KE0_DEFAULT = 0.456
KE0_MAP = 0.1

DT = 5.0             # seconds between infusion-rate decisions (Sec. 2.3.2)


@dataclass
class Patient:
    """Randomized patient demographics (Sec. 2.1.3)."""
    sex: float       # 0 = female, 1 = male
    age: float       # years, [18, 85]
    weight: float    # kg, [40, 120]
    height: float    # cm, [150, 190]

    @staticmethod
    def sample(rng: np.random.Generator) -> "Patient":
        sex = rng.integers(0, 2)
        age = rng.uniform(18, 85)
        weight = rng.uniform(40, 120)
        height = rng.uniform(150, 190)
        return Patient(sex=sex, age=age, weight=weight, height=height)

    def lean_body_mass(self) -> float:
        """Eq. (5)-(6): James formula, sex-weighted."""
        lbm_male = 1.1 * self.weight - 128 * (self.weight / self.height) ** 2
        lbm_female = 1.07 * self.weight - 148 * (self.weight / self.height) ** 2
        return self.sex * lbm_male + (1 - self.sex) * lbm_female


@dataclass
class Targets:
    """Randomized physiological targets / safety thresholds (Sec. 2.3.3)."""
    bis_target: float
    rr_threshold: float
    map_threshold: float
    hr_threshold: float

    @staticmethod
    def sample(rng: np.random.Generator) -> "Targets":
        bis_target = rng.uniform(45, 55)
        rr_threshold = rng.uniform(8, 12)
        map_threshold = rng.uniform(60, 65)
        hr_threshold = rng.uniform(40, 50)
        return Targets(bis_target, rr_threshold, map_threshold, hr_threshold)


class PropofolPKPDEnv:
    """
    Multi-target propofol dosing environment -- INDUCTION PHASE ONLY.

    The agent has to make BIS follow the exponential reference function
    (Jarrett et al. 2022, RFe):
        RFe(t) = (B0 - Bf) * exp(-lambda * t) + Bf ,   Bf = BIS target
        lambda = (ln(B0 - Bf) - ln(eps)) / t_eps       (so RFe(t_eps) = Bf + eps)
    t is in minutes.  B0 = starting reference value (default 100 = BIS baseline).
    The episode ENDS when the reference reaches Bf + eps (time t_eps); there is
    no maintenance phase.

    State (4 * history_steps + 11 dims, all normalized to ~[0, 1]):
        [BIS, RR, MAP, HR] * history_steps  +
        [BIS_target, RR_thr, MAP_thr, HR_thr, age, sex, weight, height]  +
        [BIS_ref(t), BIS_ref(t+1), t / t_eps]

    Action (1-D): propofol infusion rate in [0, MAX_INFUSION] mg/min.

    Reward: see `_reward`, divided by REWARD_SCALE for stable TD3 training.
            `info["r_total_raw"]` holds the unscaled value.
    """

    MAX_INFUSION = 10.0              # mg/min
    MAX_RATE_STEP = 2.0              # mg/min, penalty-free change between steps
                                     # (induction needs a fast ramp-up)

    # Reward exponents (Sec. 2.2)
    C_BIS, C_RR, C_MAP, C_HR = 1.2, 2.0, 1.5, 1.5

    # --- tracking of the exponential reference ------------------------------- #
    TRACK_HALF_WIDTH = 2.0           # dead-band around the reference (BIS points)
    DEEP_PENALTY_WEIGHT = 2.0        # BIS *below* the reference band (drug overshoot)
                                     # is penalized more than BIS above it
                                     # (cf. weighted over/undershoot, alpha, in Jarrett et al.)

    # --- exponential reference function parameters ----------------------------- #
    REF_EPS = 1.0                    # RFe(t_eps) = Bf + eps
    REF_T_EPS_RANGE = (4.0, 10.0)    # minutes; sampled per episode (paper: t_eps <= 10)

    # Raw rewards can reach hundreds per step, which destabilizes the critic
    # and pushes the sigmoid actor into its flat region -> scale them down.
    REWARD_SCALE = 100.0

    # State normalization divisors.
    NORM_BIS, NORM_RR, NORM_MAP, NORM_HR = 100.0, 40.0, 150.0, 200.0
    NORM_AGE, NORM_WEIGHT, NORM_HEIGHT = 85.0, 120.0, 190.0

    def __init__(self, episode_length: int = 500, history_steps: int = 1,
                 seed: int | None = None,
                 ref_b0: float = 100.0,
                 ref_t_eps: float | None = None):
        """
        ref_b0        : B0, reference value at t = 0 (<= 100).
        ref_t_eps     : fixed t_eps in minutes; None -> random in REF_T_EPS_RANGE
                        for every episode.
        """
        self.episode_length = episode_length
        self.history_steps = max(1, history_steps)
        self.rng = np.random.default_rng(seed)

        self.ref_b0 = ref_b0
        self.ref_t_eps_fixed = ref_t_eps

        self.patient: Patient | None = None
        self.targets: Targets | None = None
        self.t = 0
        self.prev_infusion = 0.0
        self.t_eps = 1.0            # minutes
        self.ref_lambda = 0.0
        self.ref_b0_eff = ref_b0

        # PK state: [c1, c2, c3, ce_bis, ce_rr, ce_map]
        self._pk_state = np.zeros(6)
        self._history = []

    # ---- PK parameters ---------------------------------------------------- #
    def _compute_pk_rates(self):
        """Eq. (7)-(20): volumes, clearances and rate constants."""
        p = self.patient
        lbm = p.lean_body_mass()

        vol1 = 4.27
        vol2 = 18.9 - 0.391 * (p.age - 53)
        vol3 = 238.0

        cl1 = 1.89 + 0.0456 * (p.weight - 77) - 0.0681 * (lbm - 59) + 0.0264 * (p.height - 177)
        cl2 = 1.29 - 0.024 * (p.age - 53)
        cl3 = 0.836

        # guard against non-physical values from extreme random draws
        vol2 = max(vol2, 1.0)
        cl1 = max(cl1, 0.05)
        cl2 = max(cl2, 0.05)

        self._k = dict(
            k10=cl1 / vol1, k12=cl2 / vol1, k13=cl3 / vol1,
            k21=cl2 / vol2, k31=cl3 / vol3,
        )

    # ---- Exponential reference function (Jarrett et al., Eq. 9-10) --------- #
    def _setup_reference(self):
        """Compute B0, lambda and t_eps for the current episode."""
        bf = self.targets.bis_target
        eps = self.REF_EPS

        lo, hi = self.REF_T_EPS_RANGE
        self.t_eps = (float(self.ref_t_eps_fixed) if self.ref_t_eps_fixed is not None
                      else float(self.rng.uniform(lo, hi)))
        # B0 must be clearly above Bf + eps for the exponential to be defined
        self.ref_b0_eff = float(min(BIS0, max(self.ref_b0, bf + 2.0 * eps)))
        self.ref_lambda = (np.log(self.ref_b0_eff - bf) - np.log(eps)) / self.t_eps

    def _ref_bis(self, step_idx: int) -> float:
        """Reference BIS value at decision step `step_idx` (time = step_idx * DT)."""
        bf = self.targets.bis_target
        t_min = step_idx * DT / 60.0
        if t_min >= self.t_eps:
            return bf          # (only reached on the very last step)
        return float((self.ref_b0_eff - bf) * np.exp(-self.ref_lambda * t_min) + bf)

    # ---- Gym-like API ----------------------------------------------------- #
    def reset(self, seed: int | None = None) -> np.ndarray:
        if seed is not None:
            self.rng = np.random.default_rng(seed)

        self.patient = Patient.sample(self.rng)
        self.targets = Targets.sample(self.rng)
        self._compute_pk_rates()
        self._setup_reference()

        self.t = 0
        self.prev_infusion = 0.0
        self._pk_state = np.zeros(6)

        vitals = self._pd_output(add_noise=False)
        self._history = [vitals for _ in range(self.history_steps)]
        return self._build_state()

    def step(self, action: float):
        """action: infusion rate in [0, MAX_INFUSION] mg/min.
        Returns (next_state, reward, done, info)."""
        u = float(np.clip(action, 0.0, self.MAX_INFUSION))

        # Pharmacokinetics: Eq. (1)-(4), Euler integration
        c1, c2, c3, ce_bis, ce_rr, ce_map = self._pk_state
        k = self._k
        dt = DT / 60.0   # seconds -> minutes

        dc1 = -(k["k10"] + k["k12"] + k["k13"]) * c1 + k["k21"] * c2 + k["k31"] * c3 + u
        dc2 = k["k12"] * c1 - k["k21"] * c2
        dc3 = k["k13"] * c1 - k["k31"] * c3

        c1_new = c1 + dc1 * dt
        c2_new = c2 + dc2 * dt
        c3_new = c3 + dc3 * dt

        ce_bis_new = ce_bis + KE0_DEFAULT * (c1_new - ce_bis) * dt
        ce_rr_new = ce_rr + KE0_DEFAULT * (c1_new - ce_rr) * dt
        ce_map_new = ce_map + KE0_MAP * (c1_new - ce_map) * dt

        self._pk_state = np.maximum(
            np.array([c1_new, c2_new, c3_new, ce_bis_new, ce_rr_new, ce_map_new]), 0.0
        )

        vitals = self._pd_output(add_noise=True)
        self._history.append(vitals)
        self._history = self._history[-self.history_steps:]

        # The measurement belongs to the time of the NEXT decision step (t + 1),
        # so it is compared with the reference value at that time.
        reward, info = self._reward(vitals, u, step_idx=self.t + 1)

        self.prev_infusion = u
        self.t += 1
        # episode = induction only: stop when the reference has reached the target
        done = (self.t >= self.episode_length) or (self.t * DT / 60.0 >= self.t_eps)
        return self._build_state(), reward, done, info

    # ---- Pharmacodynamics: Eq. (21)-(24) ---------------------------------- #
    def _pd_output(self, add_noise: bool) -> np.ndarray:
        _, _, _, ce_bis, ce_rr, ce_map = self._pk_state
        eps = self.rng.normal(0, 0.5, size=4) if add_noise else np.zeros(4)

        bis = BIS0 * (1 + ce_bis / EC50_BIS) ** GAMMA_BIS + eps[0]
        rr = RR0 * (1 + ce_rr / EC50_RR) ** GAMMA_RR + eps[1]

        map_ratio = ce_map ** GAMMA_MAP / (EC50_MAP ** GAMMA_MAP + ce_map ** GAMMA_MAP + 1e-8)
        map_ = MAP0 * (1 - E_MAX_MAP * map_ratio) + eps[2]
        hr = map_ / (SV * TPR) + eps[3]

        return np.array([
            float(np.clip(bis, 0.0, 100.0)),
            float(np.clip(rr, 0.0, 40.0)),
            float(np.clip(map_, 20.0, 150.0)),
            float(np.clip(hr, 20.0, 200.0)),
        ])

    # ---- Reward (induction: track the exponential reference) --------------- #
    def _reward(self, vitals: np.ndarray, u: float, step_idx: int):
        """
        R_total = R_BIS + R_RR + R_MAP + R_HR + R_infusion

        R_BIS : follow the exponential reference.  e = BIS - ref(t), w = TRACK_HALF_WIDTH
            |e| <= w                          ->  R_BIS = 0
            e >  +w  (BIS above reference)    ->  R_BIS = -(e - w)^C_BIS
            e <  -w  (BIS below reference)    ->  R_BIS = -W_DEEP * (-e - w)^C_BIS
        R_RR, R_MAP, R_HR : safety thresholds (0 if above threshold, else -|diff|^C)
        R_infusion        : -max(0, |u - u_prev| - MAX_RATE_STEP)
        """
        bis, rr, map_, hr = vitals
        t = self.targets

        bis_ref = self._ref_bis(step_idx)
        w = self.TRACK_HALF_WIDTH
        err = bis - bis_ref
        if abs(err) <= w:
            r_bis = 0.0
        elif err > w:                                       # BIS above reference
            r_bis = -((err - w) ** self.C_BIS)
        else:                                               # BIS below reference (too deep)
            r_bis = -self.DEEP_PENALTY_WEIGHT * ((-err - w) ** self.C_BIS)

        def threshold_reward(value, threshold, exponent):
            return 0.0 if value >= threshold else -(abs(value - threshold) ** exponent)

        r_rr = threshold_reward(rr, t.rr_threshold, self.C_RR)
        r_map = threshold_reward(map_, t.map_threshold, self.C_MAP)
        r_hr = threshold_reward(hr, t.hr_threshold, self.C_HR)

        # Infusion smoothness penalty (Eq. 29)
        r_infusion = -max(0.0, abs(u - self.prev_infusion) - self.MAX_RATE_STEP)

        r_total_raw = r_bis + r_rr + r_map + r_hr + r_infusion
        r_total = r_total_raw / self.REWARD_SCALE

        info = dict(
            bis=bis, rr=rr, map=map_, hr=hr, bis_ref=bis_ref,
            r_bis=r_bis, r_rr=r_rr, r_map=r_map, r_hr=r_hr, r_infusion=r_infusion,
            r_total_raw=r_total_raw,
        )
        return r_total, info

    # ---- State (Sec. 2.3.1) ----------------------------------------------- #
    def _build_state(self) -> np.ndarray:
        p, t = self.patient, self.targets

        static = np.array([
            t.bis_target / self.NORM_BIS,
            t.rr_threshold / self.NORM_RR,
            t.map_threshold / self.NORM_MAP,
            t.hr_threshold / self.NORM_HR,
            p.age / self.NORM_AGE,
            p.sex,
            p.weight / self.NORM_WEIGHT,
            p.height / self.NORM_HEIGHT,
        ])

        # reference information: where BIS should be now / next step, and progress in induction
        ref_feat = np.array([
            self._ref_bis(self.t) / self.NORM_BIS,
            self._ref_bis(self.t + 1) / self.NORM_BIS,
            min(1.0, self.t * DT / 60.0 / self.t_eps),
        ])

        norm = np.array([self.NORM_BIS, self.NORM_RR, self.NORM_MAP, self.NORM_HR])
        vitals_stack = np.concatenate([v / norm for v in self._history])
        return np.concatenate([vitals_stack, static, ref_feat]).astype(np.float32)

    @property
    def state_dim(self) -> int:
        return 4 * self.history_steps + 8 + 3

    @property
    def action_dim(self) -> int:
        return 1


In [ ]:
# =========================================================================== #
# 2) Replay buffer
# =========================================================================== #

class ReplayBuffer:
    def __init__(self, state_dim: int, action_dim: int, capacity: int = 200_000):
        self.capacity = capacity
        self.ptr = 0
        self.size = 0

        self.state = np.zeros((capacity, state_dim), dtype=np.float32)
        self.action = np.zeros((capacity, action_dim), dtype=np.float32)
        self.reward = np.zeros((capacity, 1), dtype=np.float32)
        self.next_state = np.zeros((capacity, state_dim), dtype=np.float32)
        self.done = np.zeros((capacity, 1), dtype=np.float32)

    def add(self, s, a, r, s2, d):
        self.state[self.ptr] = s
        self.action[self.ptr] = a
        self.reward[self.ptr] = r
        self.next_state[self.ptr] = s2
        self.done[self.ptr] = float(d)

        self.ptr = (self.ptr + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size: int):
        idx = np.random.randint(0, self.size, size=batch_size)
        return (self.state[idx], self.action[idx], self.reward[idx],
                self.next_state[idx], self.done[idx])

    def __len__(self):
        return self.size


In [ ]:
# =========================================================================== #
# 3) TD3
# =========================================================================== #

class Actor(nn.Module):
    def __init__(self, state_dim: int, action_dim: int, max_action: float, hidden: int = 64):
        super().__init__()
        self.l1 = nn.Linear(state_dim, hidden)
        self.l2 = nn.Linear(hidden, hidden)
        self.l3 = nn.Linear(hidden, action_dim)
        self.max_action = max_action

    def forward(self, state):
        x = F.relu(self.l1(state))
        x = F.relu(self.l2(x))
        # sigmoid scaled by the maximum infusion rate (Sec. 2.3.3)
        return torch.sigmoid(self.l3(x)) * self.max_action


class Critic(nn.Module):
    """Twin Q-networks in one module."""

    def __init__(self, state_dim: int, action_dim: int, hidden: int = 64):
        super().__init__()
        self.l1 = nn.Linear(state_dim + action_dim, hidden)
        self.l2 = nn.Linear(hidden, hidden)
        self.l3 = nn.Linear(hidden, 1)

        self.l4 = nn.Linear(state_dim + action_dim, hidden)
        self.l5 = nn.Linear(hidden, hidden)
        self.l6 = nn.Linear(hidden, 1)

    def forward(self, state, action):
        sa = torch.cat([state, action], dim=1)

        q1 = F.relu(self.l1(sa))
        q1 = F.relu(self.l2(q1))
        q1 = self.l3(q1)

        q2 = F.relu(self.l4(sa))
        q2 = F.relu(self.l5(q2))
        q2 = self.l6(q2)
        return q1, q2

    def q1(self, state, action):
        sa = torch.cat([state, action], dim=1)
        q1 = F.relu(self.l1(sa))
        q1 = F.relu(self.l2(q1))
        return self.l3(q1)


class TD3:
    def __init__(
        self,
        state_dim: int,
        action_dim: int,
        max_action: float,
        actor_lr: float = 1e-4,
        critic_lr: float = 1e-3,
        gamma: float = 0.99,
        tau: float = 0.005,
        policy_noise: float = 0.2,
        noise_clip: float = 0.5,
        policy_freq: int = 2,
    ):
        self.actor = Actor(state_dim, action_dim, max_action).to(device)
        self.actor_target = copy.deepcopy(self.actor)
        self.actor_optimizer = torch.optim.Adam(self.actor.parameters(), lr=actor_lr)

        self.critic = Critic(state_dim, action_dim).to(device)
        self.critic_target = copy.deepcopy(self.critic)
        self.critic_optimizer = torch.optim.Adam(self.critic.parameters(), lr=critic_lr)

        self.max_action = max_action
        self.gamma = gamma
        self.tau = tau
        self.policy_noise = policy_noise
        self.noise_clip = noise_clip
        self.policy_freq = policy_freq
        self.total_it = 0

    def select_action(self, state: np.ndarray, noise_std: float = 0.0) -> np.ndarray:
        state_t = torch.as_tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
        with torch.no_grad():
            action = self.actor(state_t).cpu().numpy().flatten()
        if noise_std > 0:
            action = action + np.random.normal(0, noise_std, size=action.shape)
        return np.clip(action, 0.0, self.max_action)

    def train_step(self, replay_buffer, batch_size: int = 256):
        self.total_it += 1

        s, a, r, s2, d = replay_buffer.sample(batch_size)
        state = torch.as_tensor(s, dtype=torch.float32, device=device)
        action = torch.as_tensor(a, dtype=torch.float32, device=device)
        reward = torch.as_tensor(r, dtype=torch.float32, device=device)
        next_state = torch.as_tensor(s2, dtype=torch.float32, device=device)
        done = torch.as_tensor(d, dtype=torch.float32, device=device)

        with torch.no_grad():
            noise = (torch.randn_like(action) * self.policy_noise).clamp(
                -self.noise_clip, self.noise_clip)
            next_action = (self.actor_target(next_state) + noise).clamp(0.0, self.max_action)

            target_q1, target_q2 = self.critic_target(next_state, next_action)
            target_q = torch.min(target_q1, target_q2)
            target_q = reward + (1 - done) * self.gamma * target_q

        current_q1, current_q2 = self.critic(state, action)
        critic_loss = F.mse_loss(current_q1, target_q) + F.mse_loss(current_q2, target_q)

        self.critic_optimizer.zero_grad()
        critic_loss.backward()
        # Gradient clipping keeps a single large-TD-error batch from pushing the
        # actor's pre-sigmoid output into the saturated (zero-gradient) region.
        torch.nn.utils.clip_grad_norm_(self.critic.parameters(), max_norm=10.0)
        self.critic_optimizer.step()

        actor_loss = None
        if self.total_it % self.policy_freq == 0:
            actor_loss = -self.critic.q1(state, self.actor(state)).mean()

            self.actor_optimizer.zero_grad()
            actor_loss.backward()
            torch.nn.utils.clip_grad_norm_(self.actor.parameters(), max_norm=10.0)
            self.actor_optimizer.step()

            self._soft_update(self.actor, self.actor_target)
            self._soft_update(self.critic, self.critic_target)

        return critic_loss.item(), (actor_loss.item() if actor_loss is not None else None)

    def _soft_update(self, source: nn.Module, target: nn.Module):
        for p, tp in zip(source.parameters(), target.parameters()):
            tp.data.copy_(self.tau * p.data + (1 - self.tau) * tp.data)

    def save(self, path: str):
        torch.save(
            dict(
                actor=self.actor.state_dict(),
                critic=self.critic.state_dict(),
                actor_optimizer=self.actor_optimizer.state_dict(),
                critic_optimizer=self.critic_optimizer.state_dict(),
                total_it=self.total_it,
            ),
            path,
        )

    def load(self, path: str):
        ckpt = torch.load(path, map_location=device)
        self.actor.load_state_dict(ckpt["actor"])
        self.critic.load_state_dict(ckpt["critic"])
        self.actor_target.load_state_dict(self.actor.state_dict())
        self.critic_target.load_state_dict(self.critic.state_dict())


In [ ]:
# =========================================================================== #
# 4) Training
# =========================================================================== #

def set_global_seed(seed: int) -> None:
    """Seed python / numpy / torch RNGs (the env owns its own RNG)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False


def train(
    episodes: int = 1000,
    episode_length: int = 120,
    history_steps: int = 1,
    batch_size: int = 256,
    exploration_noise: float = 0.5,
    exploration_noise_final: float = 0.05,
    exploration_decay_frac: float = 700,
    seed: int = 0,
    out_path: str = "td3_propofol.pt",
    log_every: int = 10,
    log_path: str = "training_log.npz",
    env_kwargs: dict | None = None,
):
    """Train TD3. Exploration noise (mg/min) decays linearly from
    `exploration_noise` to `exploration_noise_final` over the first
    `exploration_decay_frac` of training, then stays constant.
    Always records reward history and start/median/final trajectories."""
    set_global_seed(seed)

    env = PropofolPKPDEnv(episode_length=episode_length, history_steps=history_steps,
                          seed=seed, **(env_kwargs or {}))
    max_action = env.MAX_INFUSION

    agent = TD3(env.state_dim, env.action_dim, max_action)
    buffer = ReplayBuffer(env.state_dim, env.action_dim, capacity=episodes * episode_length)

    episode_rewards = []

    record_eps = {}
    for idx, label in ((0, "start"), (episodes // 2, "median"), (episodes - 1, "final")):
        record_eps[idx] = label
    trajectories = {}

    decay_episodes = max(1, exploration_decay_frac)

    for ep in range(episodes):
        state = env.reset(seed=seed + ep)
        ep_reward = 0.0

        decay_frac = min(1.0, ep / decay_episodes)
        noise_std = exploration_noise + decay_frac * (exploration_noise_final - exploration_noise)

        capture = ep in record_eps
        if capture:
            traj = dict(infusion=[], bis=[], bis_ref=[], rr=[], map=[], hr=[])

        for _ in range(episode_length):
            action = agent.select_action(state, noise_std=noise_std)

            next_state, reward, done, info = env.step(action[0])
            buffer.add(state, action, reward, next_state, done)

            if capture:
                traj["infusion"].append(float(np.clip(action[0], 0.0, max_action)))
                traj["bis"].append(info["bis"])
                traj["bis_ref"].append(info["bis_ref"])
                traj["rr"].append(info["rr"])
                traj["map"].append(info["map"])
                traj["hr"].append(info["hr"])

            state = next_state
            ep_reward += reward

            if len(buffer) >= batch_size:
                agent.train_step(buffer, batch_size=batch_size)

            if done:
                break

        if capture:
            traj = {k: np.array(v) for k, v in traj.items()}
            traj["bis_target"] = env.targets.bis_target
            traj["t_eps_min"] = env.t_eps
            traj["rr_threshold"] = env.targets.rr_threshold
            traj["map_threshold"] = env.targets.map_threshold
            traj["hr_threshold"] = env.targets.hr_threshold
            trajectories[record_eps[ep]] = traj

        episode_rewards.append(ep_reward)

        if (ep + 1) % log_every == 0:
            recent = np.mean(episode_rewards[-log_every:])
            print(f"Episode {ep + 1:5d}/{episodes} | "
                  f"reward (last {log_every}): {recent:10.1f} | noise_std: {noise_std:.3f}")

    agent.save(out_path)
    print(f"Saved trained agent to {out_path}")

    np.savez(log_path, episode_rewards=np.array(episode_rewards),
             trajectories=np.array(trajectories, dtype=object))
    print(f"Saved training log to {log_path}")

    return agent, episode_rewards, trajectories

In [ ]:
# =========================================================================== #
# 5) Plotting (Figure 3 reproduction)
# =========================================================================== #

COLORS = dict(bis="tab:blue", rr="tab:green", map="tab:red", hr="tab:purple", infusion="black")


def moving_average(x: np.ndarray, window: int = 25) -> np.ndarray:
    if len(x) < window:
        return x.copy()
    kernel = np.ones(window) / window
    padded = np.concatenate([np.full(window - 1, x[0]), x])
    return np.convolve(padded, kernel, mode="valid")


def _plot_episode_panel(ax, traj: dict, title: str):
    t = np.arange(len(traj["bis"]))

    ax.plot(t, traj["infusion"], color=COLORS["infusion"], linestyle="--", label="Infusion Rate")
    ax.plot(t, traj["bis_ref"], color="tab:orange", linestyle="--", linewidth=1.8,
            label="BIS reference (RFe)")
    ax.plot(t, traj["bis"], color=COLORS["bis"], label="BIS")
    ax.plot(t, traj["rr"], color=COLORS["rr"], label="RR")
    ax.plot(t, traj["map"], color=COLORS["map"], label="MAP")
    ax.plot(t, traj["hr"], color=COLORS["hr"], label="HR")

    bis_target = traj["bis_target"]

    for key, thr_key, name in (("bis", "bis_target", "Target BIS"),
                               ("rr", "rr_threshold", "Threshold RR"),
                               ("map", "map_threshold", "Threshold MAP"),
                               ("hr", "hr_threshold", "Threshold HR")):
        ax.axhline(traj[thr_key], color=COLORS[key], linestyle=":", linewidth=1,
                   label=f"{name}: {traj[thr_key]:.0f}")

    ax.set_title(title)
    ax.set_xlabel("Time Step (5 s)")
    ax.set_ylabel("Value")
    ax.set_ylim(0, 100)
    ax.legend(fontsize=6, loc="upper right", ncol=1)


def plot_figure3(episode_rewards, trajectories: dict, save_path: str = "figure3_reproduction.png",
                 smoothing_window: int = 25, show: bool = False):
    episode_rewards = np.asarray(episode_rewards, dtype=float)
    smoothed = moving_average(episode_rewards, window=smoothing_window)

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    ax = axes[0, 0]
    ax.plot(episode_rewards, color="tab:blue", alpha=0.5, linewidth=0.8, label="Original reward")
    ax.plot(smoothed, color="tab:orange", linewidth=2, label="Smoothed reward")
    ax.set_title("Total Rewards Per Episode")
    ax.set_xlabel("Episode")
    ax.set_ylabel("Total Reward")
    ax.legend(fontsize=8)

    _plot_episode_panel(axes[0, 1], trajectories["start"], "Infusion and Outputs in Starting Episode")
    _plot_episode_panel(axes[1, 0], trajectories["median"], "Infusion and Outputs in Median Episode")
    _plot_episode_panel(axes[1, 1], trajectories["final"], "Infusion and Outputs in Last Episode")

    fig.tight_layout()
    fig.savefig(save_path, dpi=200)
    print(f"Saved Figure 3 reproduction to {save_path}")

    if show:
        plt.show()
    else:
        plt.close(fig)
    return fig


def load_and_plot(log_path: str, save_path: str = "figure3_reproduction.png", show: bool = False):
    data = np.load(log_path, allow_pickle=True)
    return plot_figure3(data["episode_rewards"], data["trajectories"].item(),
                        save_path=save_path, show=show)

In [ ]:
# =========================================================================== #
# 6) Evaluation (tracking of the exponential reference during induction)
# =========================================================================== #

def evaluate(
    agent: TD3,
    n_patients: int = 100,
    episode_length: int = 120,
    history_steps: int = 1,
    seed: int = 1234,
    env_kwargs: dict | None = None,
):
    """Returns (mdpe, mdape, extra). PE = (BIS - reference) / reference * 100.

    extra (per patient):
        rmse_ref    : RMSE between BIS and the reference [BIS points]
        max_abs_err : largest |BIS - reference|
        min_bis_gap : lowest BIS - (target - 5); negative = deeper than the target zone
        final_err   : BIS - target at the end of induction
    """
    env = PropofolPKPDEnv(episode_length=episode_length, history_steps=history_steps,
                          seed=seed, **(env_kwargs or {}))

    mdpe_list, mdape_list = [], []
    extra = {k: [] for k in ("rmse_ref", "max_abs_err", "min_bis_gap", "final_err")}

    for i in range(n_patients):
        state = env.reset(seed=seed + i)

        bis_t, ref_t = [], []
        for _ in range(episode_length):
            action = agent.select_action(state, noise_std=0.0)
            state, _, done, info = env.step(action[0])
            bis_t.append(info["bis"]); ref_t.append(info["bis_ref"])
            if done:
                break

        bis_t, ref_t = np.asarray(bis_t), np.asarray(ref_t)
        target = env.targets.bis_target

        pe = (bis_t - ref_t) / ref_t * 100.0
        mdpe_list.append(np.median(pe))
        mdape_list.append(np.median(np.abs(pe)))
        extra["rmse_ref"].append(np.sqrt(np.mean((bis_t - ref_t) ** 2)))
        extra["max_abs_err"].append(np.max(np.abs(bis_t - ref_t)))
        extra["min_bis_gap"].append(bis_t.min() - (target - 5.0))
        extra["final_err"].append(bis_t[-1] - target)

    mdpe = np.array(mdpe_list)
    mdape = np.array(mdape_list)
    extra = {k: np.array(v, dtype=float) for k, v in extra.items()}

    ms = lambda x: f"{np.mean(x):.2f} ({np.std(x):.2f})"
    print(f"MDPE  vs reference [%]       : {ms(mdpe)}")
    print(f"MDAPE vs reference [%]       : {ms(mdape)}")
    print(f"RMSE to reference [BIS]      : {ms(extra['rmse_ref'])}")
    print(f"Max |error| [BIS]            : {ms(extra['max_abs_err'])}")
    print(f"Lowest BIS - (target-5)      : {ms(extra['min_bis_gap'])}")
    print(f"BIS - target at end          : {ms(extra['final_err'])}")
    return mdpe, mdape, extra


In [ ]:
# =========================================================================== #
# 7) Configuration  (edit these values, then run the file)
# =========================================================================== #

# --- GitHub ----------------------------------------------------------------
GITHUB_REPO   = "nazaninbondarian/PHD_Vital"                   # private repo
REPO_SUBDIR   = "Third_Steps/Exponential1/Part1"      # folder inside the repo
LOCAL_DIR     = "PHD_Vital"                                  # local clone folder name
GIT_USER_NAME = "nazaninbondarian"                           # commit author

# The token is read from Colab Secrets (key icon in the left sidebar).
# Secret name must be exactly GITHUB_TOKEN, with "Notebook access" turned ON.

# --- What to run -------------------------------------------------------------
RUN_TRAIN    = True     # train + save checkpoint & training log (old files are deleted first)
RUN_PLOT     = True     # draw Figure 3 from the training log
RUN_EVALUATE = True     # MDPE / MDAPE evaluation of the saved checkpoint

# --- Training ------------------------------------------------------------------
EPISODES       = 1000    # paper default; use e.g. 50 for a quick smoke test
EPISODE_LENGTH = 120       # max steps (5 s each); episode ends at t_eps (<= 10 min = 120 steps)
HISTORY_STEPS  = 1       # 1 = base TD3 (12-D state); 5 = "TD3 (5 steps)" (28-D state)
SEED           = 0

# --- Exponential reference function (induction) -----------------------------
REF_B0    = 100.0   # B0: reference value at t = 0 (<= 100, the BIS baseline)
REF_T_EPS = None    # t_eps in minutes (reference reaches Bf+1); None -> random 4-10 min per episode

ENV_KWARGS = dict(ref_b0=REF_B0, ref_t_eps=REF_T_EPS)

CHECKPOINT_PATH = "td3_propofol.pt"
LOG_PATH        = "training_log.npz"
FIGURE_PATH     = "figure3_reproduction.png"

# --- Evaluation ------------------------------------------------------------------
N_PATIENTS          = 100
EVAL_EPISODE_LENGTH = 120


In [ ]:
# =========================================================================== #
# 8) GitHub helpers
# =========================================================================== #

def get_github_token() -> str:
    """Colab Secret `GITHUB_TOKEN` -> env var GITHUB_TOKEN -> interactive prompt."""
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
        if token:
            return token
    except Exception as e:   # not on Colab, secret missing, or notebook access off
        print(f"Colab secret GITHUB_TOKEN not available ({type(e).__name__}); trying fallbacks.")
    return os.environ.get("GITHUB_TOKEN") or getpass.getpass("GitHub Token (repo scope): ")


def commit_and_push(repo_dir: str, message: str, token: str) -> None:
    """Stage everything in the repo, commit and push (token is masked in any output)."""
    def clean(text: str) -> str:
        return text.replace(token, "***") if token else text

    git = ["git", "-C", repo_dir]
    subprocess.run(git + ["config", "user.email", f"{GIT_USER_NAME}@users.noreply.github.com"], check=True)
    subprocess.run(git + ["config", "user.name", GIT_USER_NAME], check=True)
    subprocess.run(git + ["add", "-A"], check=True)

    commit = subprocess.run(git + ["commit", "-m", message], capture_output=True, text=True)
    print(clean(commit.stdout or commit.stderr))

    push = subprocess.run(git + ["push"], capture_output=True, text=True)
    print(clean(push.stdout or push.stderr))
    if push.returncode != 0:
        raise RuntimeError("git push failed:\n" + clean(push.stderr))

    print(f"Pushed. Files are at: https://github.com/{GITHUB_REPO}/tree/main/{REPO_SUBDIR}")

In [ ]:
# =========================================================================== #
# 9) Main
# =========================================================================== #

def main():
    print(f"Using device: {device}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)} | CUDA {torch.version.cuda}")

    # ---- clone / pull the repo -------------------------------------------- #
    token = get_github_token()
    remote_url = f"https://{token}@github.com/{GITHUB_REPO}.git"
    repo_dir = os.path.abspath(LOCAL_DIR)

    if not os.path.exists(repo_dir):
        subprocess.run(["git", "clone", remote_url, repo_dir], check=True)
    else:
        subprocess.run(["git", "-C", repo_dir, "remote", "set-url", "origin", remote_url], check=True)
        subprocess.run(["git", "-C", repo_dir, "pull"], check=True)

    project_dir = os.path.join(repo_dir, REPO_SUBDIR)
    os.makedirs(project_dir, exist_ok=True)

    # keep a copy of this script next to the results
    script = globals().get("__file__")
    if script and os.path.isfile(script):
        dest = os.path.join(project_dir, os.path.basename(script))
        if os.path.abspath(script) != os.path.abspath(dest):
            shutil.copy(script, dest)

    os.chdir(project_dir)
    print("Working directory is now:", os.getcwd())

    # ---- train ---------------------------------------------------------------- #
    if RUN_TRAIN:
        for fname in (CHECKPOINT_PATH, LOG_PATH, FIGURE_PATH):
            if os.path.exists(fname):
                os.remove(fname)
                print(f"Removed old file: {fname}")

        train(
            episodes=EPISODES,
            episode_length=EPISODE_LENGTH,
            history_steps=HISTORY_STEPS,
            seed=SEED,
            out_path=CHECKPOINT_PATH,
            log_path=LOG_PATH,
            env_kwargs=ENV_KWARGS,
        )

    # ---- plot ----------------------------------------------------------------- #
    if RUN_PLOT:
        if os.path.exists(LOG_PATH):
            load_and_plot(LOG_PATH, save_path=FIGURE_PATH)
        else:
            print(f"{LOG_PATH} not found in {project_dir} -- run with RUN_TRAIN = True first.")

    if RUN_TRAIN or RUN_PLOT:
        msg = ("Train TD3 propofol induction agent (exp. reference): checkpoint + log + figure"
               if RUN_TRAIN else "Re-plot Figure 3 from training log")
        commit_and_push(repo_dir, msg, token)

    # ---- evaluate ------------------------------------------------------------- #
    if RUN_EVALUATE:
        dummy_env = PropofolPKPDEnv(history_steps=HISTORY_STEPS, **ENV_KWARGS)
        agent = TD3(dummy_env.state_dim, dummy_env.action_dim, dummy_env.MAX_INFUSION)
        agent.load(CHECKPOINT_PATH)

        mdpe, mdape, extra = evaluate(
            agent,
            n_patients=N_PATIENTS,
            episode_length=EVAL_EPISODE_LENGTH,
            history_steps=HISTORY_STEPS,
            env_kwargs=ENV_KWARGS,
        )

        np.savez("evaluation_results.npz", mdpe=mdpe, mdape=mdape, **extra,
                 n_patients=N_PATIENTS, episode_length=EVAL_EPISODE_LENGTH)

        with open("evaluation_summary.txt", "w") as f:
            f.write(f"Checkpoint: {CHECKPOINT_PATH}\n")
            f.write(f"n_patients={N_PATIENTS}, episode_length={EVAL_EPISODE_LENGTH}, "
                    f"history_steps={HISTORY_STEPS}, ref_b0={REF_B0}, ref_t_eps={REF_T_EPS}\n")
            f.write(f"MDPE vs reference : {mdpe.mean():.2f} ({mdpe.std():.2f})\n")
            f.write(f"MDAPE vs reference: {mdape.mean():.2f} ({mdape.std():.2f})\n")
            for k, v in extra.items():
                f.write(f"{k}: {v.mean():.2f} ({v.std():.2f})\n")
        print("Saved evaluation_results.npz and evaluation_summary.txt")

        commit_and_push(repo_dir, "Evaluate TD3 propofol agent: save MDPE/MDAPE results", token)


if __name__ == "__main__":
    main()